# 🔄 Основни токови рада агента са Microsoft Foundry (Python)

## 📋 Туторијал за оркестрацију токова рада

Овај нотебук представља снажне могућности **Workflow Builder** у Microsoft Agent Framework-у. Научите како да креирате сложене, вишестепене токове рада агента који могу да управљају комплексним пословним процесима и беспрекорно координишу више AI операција.

> **Напомена о миграцији:** Овај пример је раније користио GitHub Models. GitHub Models се више не користи (повољавање у јулу 2026), па сада користи **Microsoft Foundry** преко `FoundryChatClient`, који циља на Azure OpenAI **Responses API**.

## 🎯 Циљеви учења

### 🏗️ **Архитектура тока рада**
- **Workflow Builder**: Дизајнирање и оркестрација сложених вишестепених процеса
- **Извршење на основу догађаја**: Обрада догађаја тока рада и промена стања
- **Визуелни дизајн тока рада**: Креирање и визуализација структура тока рада
- **Интеграција Microsoft Foundry**: Коришћење AI модела у контексту тока рада

### 🔄 **Оркестрација процеса**
- **Секвенцијалне операције**: Повезивање више агенатских задатака у логичном редоследу
- **Условна логика**: Имплементација тачака одлучивања и разгранавања токова рада
- **Обрада грешака**: Робусно опорављање од грешака и отпорност тока рада
- **Управљање стањем**: Праћење и управљање стањем извршења тока рада

### 📊 **Обрасци тока рада у предузећима**
- **Аутоматизација пословних процеса**: Аутоматизација сложених организационих токова рада
- **Координација више агената**: Координација више специјализованих агената
- **Скалабилно извршење**: Дизајн токова рада за операције на нивоу предузећа
- **Надгледање и посматрање**: Праћење перформанси и резултата тока рада

## ⚙️ Захтеви и подешавање

### 📦 **Потребне зависности**

Инсталирајте Agent Framework са могућностима тока рада:

```bash
pip install agent-framework -U
```

### 🔑 **Конфигурација Microsoft Foundry**

Пријавите се помоћу Azure CLI (`az login`) да би `AzureCliCredential` могао аутентификовати, затим подесите детаље вашег Microsoft Foundry пројекта.

**Подешавање окружења (.env фајл):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

### 🏢 **Коришћење у предузећима**

**Примери пословних процеса:**
- **Приступ корисницима**: Вишестепена проверa и процес подешавања
- **Цењски проток садржаја**: Аутоматизовано стварање, преглед и објављивање садржаја
- **Обрада података**: ETL токови рада са AI покретаном трансформацијом
- **Контрола квалитета**: Аутоматизовани процеси тестирања и валидације

**Предности тока рада:**
- 🎯 **Поузданост**: Детерминисано извршење са опоравком од грешака
- 📈 **Скалабилност**: Управљање аутоматизацијом процеса великог обима
- 🔍 **Посматрање**: Комплетан запис активности и надгледање
- 🔧 **Одрживост**: Визуелни дизајн и модуларни елементи

## 🎨 Обрасци дизајна тока рада

### Основна структура тока рада
```mermaid
graph TD
    A[Почетак] --> B[Задатак агентa 1]
    B --> C{Тачка одлуке}
    C -->|Успех| D[Задатак агентa 2]
    C -->|Неуспех| E[Обрада грешке]
    D --> F[Крај]
    E --> F
```

**Кључне компоненте:**
- **WorkflowBuilder**: Главни мотор за оркестрацију
- **WorkflowEvent**: Обрада догађаја и комуникација
- **WorkflowViz**: Визуелна репрезентација и дебаговање тока рада

Хајде да креирамо ваш први интелигентни ток рада! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
